In [7]:
!nvidia-smi


Mon Apr 13 15:16:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          On  |   00000000:98:00.0 Off |                    0 |
| N/A   49C    P0             66W /  300W |    1381MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import os
import subprocess
from transformers import AutoTokenizer

BASE_MODEL_PATH = 'gpt2-large'
INPUT_FILE = './workspace/inference_data/distillation_prompts.json'

DATASET = 'gpt2-large'
DATA_PATH = f'./workspace/hmm_data/{DATASET}'
MODEL_ID = f'chmm_{DATASET}'
MODEL_PATH = f'./workspace/models/{MODEL_ID}'
LOG_FILE = f'./workspace/logs/{MODEL_ID}_log.txt'

TOTAL_SAMPLES = 10_000_000
CHUNK_SIZE = 100_000
TOTAL_CHUNKS = TOTAL_SAMPLES // CHUNK_SIZE
DEV_SIZE = 20_000
SEQUENCE_LEN = 32

CUDA_CORES = '0'
SAMPLE_BATCH_SIZE = 256
TRAIN_BATCH_SIZE = 4096
TRAIN_EVAL_SIZE = 20_000
SAVE_PER_STEP = 50
INIT_CHUNK_COUNT = 16
INIT_CONTEXT = 'both'

# Tuned restart recipe: 3 full passes, 25 chunks per EM update,
# with paper-style online EM count averaging.
EM_SCHEDULE = '12,25'
ONLINE_COUNT_DECAY = 0.90

# Broader clone schedule than the first run, while keeping max_clones=4.
CLONE_TOP4 = 512
CLONE_TOP2 = 4096

STORAGE_DTYPE = 'bfloat16'
PSEUDOCOUNT = 1e-3
ROW_CHUNK_SIZE = 1024

RUN_COMMANDS = True

os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs('./workspace/logs', exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)
VOCAB_SIZE = tokenizer.vocab_size
EOS_TOKEN_ID = tokenizer.eos_token_id

print(f'vocab_size={VOCAB_SIZE}, eos_token_id={EOS_TOKEN_ID}, total_chunks={TOTAL_CHUNKS}')
print(f'clone_top4={CLONE_TOP4}, clone_top2={CLONE_TOP2}, init_chunk_count={INIT_CHUNK_COUNT}')
print(f'em_schedule={EM_SCHEDULE}, online_count_decay={ONLINE_COUNT_DECAY}')


vocab_size=50257, eos_token_id=50256, total_chunks=100
clone_top4=512, clone_top2=4096, init_chunk_count=16
em_schedule=12,25, online_count_decay=0.9


In [3]:
def run_cmd(cmd: str):
    print(cmd)
    if RUN_COMMANDS:
        subprocess.run(cmd, shell=True, check=True)


In [4]:
def schedule_chunk_uses(em_schedule: str) -> int:
    total = 0
    for part in em_schedule.split(';'):
        repeats, chunk_count = part.split(',')
        total += int(repeats) * int(chunk_count)
    return total

def build_init_cmd() -> str:
    return f'''CUDA_VISIBLE_DEVICES={CUDA_CORES} python train_chmm.py \
    --model_path {MODEL_PATH} \
    --checkpoint 0 --init_only \
    --data_path {DATA_PATH} --dataset {DATASET} --total_chunks {TOTAL_CHUNKS} \
    --sample_length {SEQUENCE_LEN} \
    --tokenizer_name_or_path {BASE_MODEL_PATH} \
    --batch_size {TRAIN_BATCH_SIZE} \
    --init_chunk_count {INIT_CHUNK_COUNT} --init_context {INIT_CONTEXT} \
    --clone_top4 {CLONE_TOP4} --clone_top2 {CLONE_TOP2} \
    --storage_dtype {STORAGE_DTYPE} --pseudocount {PSEUDOCOUNT} \
    --row_chunk_size {ROW_CHUNK_SIZE} --device cuda \
    --log_file {LOG_FILE}'''.strip()

def build_train_cmd() -> str:
    total_uses = schedule_chunk_uses(EM_SCHEDULE)
    assert total_uses % TOTAL_CHUNKS == 0, (
        f'schedule {EM_SCHEDULE} must cover an integer number of full passes'
    )
    return f'''CUDA_VISIBLE_DEVICES={CUDA_CORES} python train_chmm.py \
    --model_path {MODEL_PATH} \
    --checkpoint 0 --save_per_step {SAVE_PER_STEP} \
    --data_path {DATA_PATH} --dataset {DATASET} --total_chunks {TOTAL_CHUNKS} \
    --sample_length {SEQUENCE_LEN} \
    --tokenizer_name_or_path {BASE_MODEL_PATH} \
    --batch_size {TRAIN_BATCH_SIZE} --train_eval_size {TRAIN_EVAL_SIZE} \
    --em_schedule "{EM_SCHEDULE}" --online_count_decay {ONLINE_COUNT_DECAY} \
    --init_context {INIT_CONTEXT} \
    --clone_top4 {CLONE_TOP4} --clone_top2 {CLONE_TOP2} \
    --storage_dtype {STORAGE_DTYPE} --pseudocount {PSEUDOCOUNT} \
    --row_chunk_size {ROW_CHUNK_SIZE} --device cuda \
    --log_file {LOG_FILE}'''.strip()

init_cmd = build_init_cmd()
train_cmd = build_train_cmd()
total_uses = schedule_chunk_uses(EM_SCHEDULE)
print(f'total_chunk_uses={total_uses}, passes={total_uses / TOTAL_CHUNKS:.1f}')
print('init_cmd:')
print(init_cmd)
print('\ntrain_cmd:')
print(train_cmd)


total_chunk_uses=300, passes=3.0
init_cmd:
CUDA_VISIBLE_DEVICES=0 python train_chmm.py     --model_path ./workspace/models/chmm_gpt2-large     --checkpoint 0 --init_only     --data_path ./workspace/hmm_data/gpt2-large --dataset gpt2-large --total_chunks 100     --sample_length 32     --tokenizer_name_or_path gpt2-large     --batch_size 4096     --init_chunk_count 16 --init_context both     --clone_top4 512 --clone_top2 4096     --storage_dtype bfloat16 --pseudocount 0.001     --row_chunk_size 1024 --device cuda     --log_file ./workspace/logs/chmm_gpt2-large_log.txt

train_cmd:
CUDA_VISIBLE_DEVICES=0 python train_chmm.py     --model_path ./workspace/models/chmm_gpt2-large     --checkpoint 0 --save_per_step 50     --data_path ./workspace/hmm_data/gpt2-large --dataset gpt2-large --total_chunks 100     --sample_length 32     --tokenizer_name_or_path gpt2-large     --batch_size 4096 --train_eval_size 20000     --em_schedule "12,25" --online_count_decay 0.9     --init_context both     --clo

In [5]:
# Run this once after deleting old CHMM checkpoints.
run_cmd(init_cmd)


CUDA_VISIBLE_DEVICES=0 python train_chmm.py     --model_path ./workspace/models/chmm_gpt2-large     --checkpoint 0 --init_only     --data_path ./workspace/hmm_data/gpt2-large --dataset gpt2-large --total_chunks 100     --sample_length 32     --tokenizer_name_or_path gpt2-large     --batch_size 4096     --init_chunk_count 16 --init_context both     --clone_top4 512 --clone_top2 4096     --storage_dtype bfloat16 --pseudocount 0.001     --row_chunk_size 1024 --device cuda     --log_file ./workspace/logs/chmm_gpt2-large_log.txt
Initializing checkpoint-0...
CHMM states=55889, max_clones=4, token_table=5.23 GiB, token_counts=10.46 GiB


init chunk 15: 100%|██████████| 25/25 [00:00<00:00, 55.58it/s]


Saved checkpoint-0 to workspace/models/chmm_gpt2-large/checkpoint-0


In [ ]:
# Run this as one uninterrupted training job from scratch.
# Because online_count_decay keeps an EMA of expected counts, do not split
# this tuned run into separate pass-2/pass-3 resume jobs.
run_cmd(train_cmd)


CUDA_VISIBLE_DEVICES=0 python train_chmm.py     --model_path ./workspace/models/chmm_gpt2-large     --checkpoint 0 --save_per_step 50     --data_path ./workspace/hmm_data/gpt2-large --dataset gpt2-large --total_chunks 100     --sample_length 32     --tokenizer_name_or_path gpt2-large     --batch_size 4096 --train_eval_size 20000     --em_schedule "12,25" --online_count_decay 0.9     --init_context both     --clone_top4 512 --clone_top2 4096     --storage_dtype bfloat16 --pseudocount 0.001     --row_chunk_size 1024 --device cuda     --log_file ./workspace/logs/chmm_gpt2-large_log.txt


em gpt2-large.train.24: 100%|██████████| 25/25 [00:22<00:00,  1.13it/s]


ckpt=25	train_step_nll_tok=4.623613	train_eval_nll_tok=4.446692	dev_nll_tok=4.936500	elapsed_s=579.4


em gpt2-large.train.49: 100%|██████████| 25/25 [00:23<00:00,  1.05it/s]


ckpt=50	train_step_nll_tok=4.939531	train_eval_nll_tok=4.453635	dev_nll_tok=4.852683	elapsed_s=569.3


...

em gpt2-large.train.75:   0%|          | 0/25 [00:00<?, ?it/s]

ckpt=175	train_step_nll_tok=4.609389	train_eval_nll_tok=4.478549	dev_nll_tok=4.763404	elapsed_s=585.5


em gpt2-large.train.99: 100%|██████████| 25/25 [00:21<00:00,  1.15it/s]


ckpt=200	train_step_nll_tok=4.606551	train_eval_nll_tok=4.485479	dev_nll_tok=4.756584	elapsed_s=613.0


em gpt2-large.train.0:  16%|█▌        | 4/25 [00:04<00:21,  1.04s/it]